In [12]:
pip install pydotplus

Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc
import seaborn as sns

"""For DT plotting"""
#from sklearn.externals.six import StringIO
from IPython.display import Image
from sklearn.tree import export_graphviz
import pydotplus

In [11]:
pip install pydotplus

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pydotplus: filename=pydotplus-2.0.2-py3-none-any.whl size=24576 sha256=533949a40a35d5cc7702dfe24a3222a36925a721fca3a5a6cec569610a0a98c9
  Stored in directory: c:\users\swapn\appdata\local\pip\cache\wheels\4a\c0\ed\a9eeeb08c3c53bb90d3822cf76557c8fdcbc349ee11a011169
Successfully built pydotplus
Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'pydotplus' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pydotplus'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [15]:
df = pd.read_csv('bank.csv')

In [16]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [17]:
# Preprocessing and EDA

In [18]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

In [22]:
#Correlation
corrmat = df.corr(numeric_only=True)
corrmat

,age,balance,day,duration,campaign,pdays,previous
age,1.000000,0.112300,-0.000762,0.000189,-0.005278,0.002774,0.020169
balance,0.112300,1.000000,0.010467,0.022436,-0.013894,0.017411,0.030805
day,-0.000762,0.010467,1.000000,-0.018511,0.137007,-0.077232,-0.058981
duration,0.000189,0.022436,-0.018511,1.000000,-0.041557,-0.027392,-0.026716
campaign,-0.005278,-0.013894,0.137007,-0.041557,1.000000,-0.102726,-0.049699
pdays,0.002774,0.017411,-0.077232,-0.027392,-0.102726,1.000000,0.507272
previous,0.020169,0.030805,-0.058981,-0.026716,-0.049699,0.507272,1.000000


In [23]:
# Label encoder
def preprocessor(df):
    
    res_df = df.copy()
    le = preprocessing.LabelEncoder()
    
    res_df['job'] = le.fit_transform(res_df['job'])
    res_df['marital'] = le.fit_transform(res_df['marital'])
    res_df['education'] = le.fit_transform(res_df['education'])
    res_df['default'] = le.fit_transform(res_df['default'])
    res_df['housing'] = le.fit_transform(res_df['housing'])
    res_df['loan'] = le.fit_transform(res_df['loan'])
    res_df['contact'] = le.fit_transform(res_df['contact'])
    res_df['month'] = le.fit_transform(res_df['month'])
    res_df['poutcome'] = le.fit_transform(res_df['poutcome'])
    res_df['deposit'] = le.fit_transform(res_df['deposit'])
    
    return res_df

In [24]:
encoded_df = preprocessor(df)

In [25]:
encoded_df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,0,1,1,0,2343,1,0,2,5,8,1042,1,-1,0,3,1
1,56,0,1,1,0,45,0,0,2,5,8,1467,1,-1,0,3,1
2,41,9,1,1,0,1270,1,0,2,5,8,1389,1,-1,0,3,1
3,55,7,1,1,0,2476,1,0,2,5,8,579,1,-1,0,3,1
4,54,0,1,2,0,184,0,0,2,5,8,673,2,-1,0,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,1,2,0,0,1,1,0,0,20,0,257,1,-1,0,3,0
11158,39,7,1,1,0,733,0,0,2,16,6,83,4,-1,0,3,0
11159,32,9,2,1,0,29,0,0,0,19,1,156,2,-1,0,3,0
11160,43,9,1,1,0,0,0,1,0,8,8,9,2,172,5,0,0


In [27]:
x = encoded_df.drop('deposit',axis=1).values
y = encoded_df['deposit']

In [28]:
x

array([[ 59,   0,   1, ...,  -1,   0,   3],
       [ 56,   0,   1, ...,  -1,   0,   3],
       [ 41,   9,   1, ...,  -1,   0,   3],
       ...,
       [ 32,   9,   2, ...,  -1,   0,   3],
       [ 43,   9,   1, ..., 172,   5,   0],
       [ 34,   9,   1, ...,  -1,   0,   3]])

In [30]:
x_train

array([[37,  9,  1, ..., -1,  0,  3],
       [44,  7,  2, ..., -1,  0,  3],
       [25,  7,  2, ..., -1,  0,  3],
       ...,
       [24,  4,  2, ..., -1,  0,  3],
       [55,  5,  1, ..., -1,  0,  3],
       [79,  5,  1, ..., -1,  0,  3]])

In [31]:
# Build decission tree

In [38]:
model_dt_2 = DecisionTreeClassifier(max_depth=2)
model_dt_2

DecisionTreeClassifier(max_depth=2)

In [40]:
#DT with max_depth 2
model_dt_2 = DecisionTreeClassifier(max_depth = 2)
model_dt_2.fit(x_train, y_train)
model_dt_2_tr_score = model_dt_2.score(x_train, y_train)
model_dt_2_te_score = model_dt_2.score(x_test, y_test)

print('Training Score', model_dt_2_tr_score)
print('Test Score', model_dt_2_te_score)

Training Score 0.7252771866950386
Test Score 0.7008508732646663


In [41]:
#DT with max_depth 4
model_dt_4 = DecisionTreeClassifier(max_depth = 4)
model_dt_4.fit(x_train, y_train)
model_dt_4_tr_score = model_dt_4.score(x_train, y_train)
model_dt_4_te_score = model_dt_4.score(x_test, y_test)

print('Training Score', model_dt_4_tr_score)
print('Test Score', model_dt_4_te_score)

Training Score 0.7944898644865046
Test Score 0.7733990147783252


In [42]:
#DT with max_depth 6
model_dt_6 = DecisionTreeClassifier(max_depth = 6)
model_dt_6.fit(x_train, y_train)
model_dt_6_tr_score = model_dt_6.score(x_train, y_train)
model_dt_6_te_score = model_dt_6.score(x_test, y_test)

print('Training Score', model_dt_6_tr_score)
print('Test Score', model_dt_6_te_score)

Training Score 0.8320080636129465
Test Score 0.812807881773399


In [43]:
#DT with max_depth 8
model_dt_8 = DecisionTreeClassifier(max_depth = 8)
model_dt_8.fit(x_train, y_train)
model_dt_8_tr_score = model_dt_8.score(x_train, y_train)
model_dt_8_te_score = model_dt_8.score(x_test, y_test)

print('Training Score', model_dt_8_tr_score)
print('Test Score', model_dt_8_te_score)

Training Score 0.8561989024526823
Test Score 0.8007165248544559


In [44]:
#DT with max_depth 21 shows overfitting
model_dt_overfit = DecisionTreeClassifier(max_depth = 21)
model_dt_overfit.fit(x_train, y_train)
model_dt_overfit_tr_score = model_dt_overfit.score(x_train, y_train)
model_dt_overfit_te_score = model_dt_overfit.score(x_test, y_test)

print('Training Score', model_dt_overfit_tr_score)
print('Test Score', model_dt_overfit_te_score)

Training Score 0.9930563332960017
Test Score 0.77384684281236


In [46]:
from io import StringIO

In [51]:
from IPython.display import Image
plt.figure(figsize = (10,8))
dot_data = StringIO()
export_graphviz(model_dt_2, out_file = dot_data, filled = True)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())

Image(graph.create_png())

InvocationException: GraphViz's executables not found

<Figure size 1000x800 with 0 Axes>